# 12 · Modelo panel — GradientBoosting global

Un unico GradientBoosting sobre todos los municipios (panel) con one-hot de municipio, usando el tablon analitico de `08_panel_features`.

- Ficha: `GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)`
- Split temporal y prediccion recursiva igual que 11
- Resultados: `results/12_modelo_panel_gbm_metrics.csv`


In [1]:
import polars as pl
import numpy as np
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("data")
RESULTS = Path("results")
MODELS = Path("../models")
RESULTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)


In [2]:
panel = pl.read_parquet(DATA / "panel_features.parquet")
meta = json.loads((DATA / "panel_metadata.json").read_text())
FEATURES = meta["features"]
TEST_START = meta["test_start"]
TRAIN_DESDE = meta.get("train_desde", 2016)
print("FEATURES:", FEATURES, "| TRAIN_DESDE:", TRAIN_DESDE)


FEATURES: ['anio', 'iph_media', 'iph_max', 'ocupacion_media', 'lluvia_anual_mm', 'lag1'] | TRAIN_DESDE: 2016


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metricas(test, pred):
    test, pred = np.asarray(test, float), np.asarray(pred, float)
    return dict(
        mae=float(mean_absolute_error(test, pred)),
        mape=float(np.mean(np.abs((test - pred) / test)) * 100),
        rmse=float(mean_squared_error(test, pred) ** 0.5),
        r2=float(r2_score(test, pred)),
    )


In [4]:
from sklearn.ensemble import GradientBoostingRegressor

pdf = panel.to_pandas()
dummies = pd.get_dummies(pdf["cod_municipio"], prefix="mun", dtype=float)
X_all = pd.concat([pdf[FEATURES], dummies], axis=1)
X_all["cod_municipio"] = pdf["cod_municipio"].values
X_all["consumo_hm3"] = pdf["consumo_hm3"].values
feature_cols = [c for c in X_all.columns if c not in ("cod_municipio", "consumo_hm3")]
print("features:", len(feature_cols))


features: 73


In [5]:
train = X_all[(X_all["anio"] >= TRAIN_DESDE) & (X_all["anio"] < TEST_START) & X_all["lag1"].notna()]
model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(train[feature_cols], train["consumo_hm3"])

def predict_recursivo_panel(train_rows, test_rows):
    last = float(train_rows["consumo_hm3"].iloc[-1])
    preds = []
    for _, row in test_rows.iterrows():
        x = {c: row[c] for c in feature_cols}
        x["lag1"] = last
        p = max(float(model.predict([list(x[f] for f in feature_cols)])[0]), 0.0)
        preds.append(p)
        last = p
    return preds

filas = []
for cod_t, g in panel.partition_by("cod_municipio", as_dict=True).items():
    cod = int(cod_t[0])
    test = g.filter(pl.col("anio") >= TEST_START)
    tr_p = X_all[(X_all["cod_municipio"] == cod) & (X_all["anio"] >= TRAIN_DESDE) & (X_all["anio"] < TEST_START) & (X_all["lag1"].notna())]
    te_p = X_all[(X_all["cod_municipio"] == cod) & (X_all["anio"] >= TEST_START)]
    pred = predict_recursivo_panel(tr_p, te_p)
    filas.append({"modelo": "gb_panel", "cod_municipio": cod,
                  **metricas(test["consumo_hm3"].to_list(), pred)})

res = pd.DataFrame(filas)
res.to_csv(RESULTS / "12_modelo_panel_gbm_metrics.csv", index=False)
print(res[["mae", "mape", "rmse", "r2"]].mean().round(3))


mae       0.157
mape     25.883
rmse      0.168
r2     -349.389
dtype: float64


In [6]:
base = pd.read_csv(RESULTS / "10_baseline_metrics.csv")
naive = base[base["modelo"] == "naive"].set_index("cod_municipio")["mape"].sort_index()
mine = res.set_index("cod_municipio")["mape"].sort_index()
print("gb_panel mejora a naive:", (mine < naive).sum(), "de", len(naive), "| MAPE",
      round(mine.mean(), 1), "vs", round(naive.mean(), 1))


gb_panel mejora a naive: 28 de 67 | MAPE 25.9 vs 11.6


**Conclusiones**

- El panel comparte senal entre municipios (one-hot, ~400 filas de train) pero pierde los efectos especificos.
- Si queda lejos del modelo por municipio, los efectos individuales dominan (cada municipio es un regimen distinto).
